---
title: "Text Classification Pipeline and Evidence Contract"
categories: [machine-learning, text-classification]
---

This course follows a detector that assigns text a score for the AI-generated class, then places that task-specific system within the broader history of text classification. A classifier can learn differences in source, topic, length, editing process, or generator instead of a stable distinction between “human” and “AI.” The same issue appears in sentiment and spam systems: measured performance is always tied to the examples, labels, and split used to establish it.

## The running system

The [source repository](https://github.com/rasbt/ai-detector-from-scratch) turns source documents into labeled examples, creates grouped train/validation/test splits, trains several classifier families, compares errors and costs, and packages selected checkpoints for local use. Here, a **group** is a set of examples tied to one underlying source or derivation. A human source and AI text derived from it belong together when splitting, so a model cannot benefit from seeing near-related content on both sides of evaluation.

```{mermaid}
flowchart LR
  A[Licensed human sources] --> C[Label and provenance audit]
  B[AI generations and edits] --> C
  C --> D[Grouped train / validation / test]
  D --> E[TF-IDF + logistic regression]
  D --> F[Prompted and fine-tuned models]
  D --> G[General-purpose decision API]
  E --> H[Quality, calibration, errors, cost, task transfer]
  F --> H
  G --> H
  H --> I[Versioned model and inference contract]
```

The diagram names the course comparisons. The data-generation path uses external services or local models; the logistic baseline can start from the published dataset and an existing checkpoint. Chapters 02–04 establish detector data and a CPU-friendly reference. Chapters 05–06 add the historical architecture map and model-family fine-tuning. Chapters 07–09 evaluate probabilities, errors, resource use, and interfaces. Chapter 10 turns the candidate-scoring design into a runnable, multi-task ModernBERT decision system and audits its task transfer.

## A short history of text classification

Before transformers, bag-of-words features made variable-length documents usable by fixed-width classifiers such as Naive Bayes, logistic regression, support vector machines, and boosted trees. Counts or TF–IDF are sparse and efficient, but a plain bag discards word order. N-grams restore short local patterns at the cost of a larger vocabulary. Chapter 04 develops the course baseline and asks learners to inspect which surface cues it uses.

Neural methods introduced dense word embeddings and ways to model sequence structure. Recurrent neural networks update a hidden state token by token; LSTMs and GRUs use gates to control what information persists. Convolutional text models learn filters over local token windows and can process those windows in parallel. Pretraining followed by task-specific fine-tuning, illustrated by ULMFiT before the transformer era, showed how transfer learning could improve a target classifier.

Transformers shifted the tradeoffs. Encoder models such as BERT are natural classification backbones; decoder models such as GPT can classify by prompting, generating a label, or using a classification head; encoder-decoder models such as T5 can emit labels as text or use a head. These are design options rather than a ladder where every newer family must win. Dataset, label quality, token limits, tuning, and evaluation protocol can matter more than architecture. Chapters 05 and 06 connect these alternatives to the detector project and capstone scoring design.

## Task-specific models and general-purpose decisions

A task-specific classifier learns a fixed label set for a defined problem. It can be fast and strong when the task repeats at scale, the labels are stable, and representative training data are available. A prompted general-purpose LLM can handle many one-off tasks, but free-form generation may add latency, cost, and parsing failures. A general-purpose decision model aims to accept instructions and candidate labels or criteria across many tasks, avoiding a new fine-tuning run for every task while returning structured decisions.

The article uses Jev as a case study for that middle point. Its API description separates three output forms: **Choice** selects one label from a candidate set; **Noul** answers one or more yes/no questions, with separate probabilities that need not sum to one; **Score** assigns an ordinal rubric value. The interface can be studied as a black box even though Jev's exact model, training data, and algorithm are not public.

The article also sketches a Jev-like transformer design: pair the task instructions and input state with each candidate description, encode each candidate-specific sequence, and score it with one shared scalar head. A softmax over the candidate scores produces the Choice distribution. Because the same head scores every candidate, the number and names of options can vary at inference time. Chapter 10 implements this design pattern using a pretrained ModernBERT encoder and training examples from multiple tasks. It is an independent educational model; it does not reproduce Jev's undisclosed internals or training recipe.

The article reports Jev sentiment results on an IMDb test set and compares them with a ModernBERT experiment, while noting run-to-run variation and that test-set inclusion in training is unknown. Those measurements are useful as a concrete comparison protocol, not a general benchmark or proof of broad task transfer. The course asks learners to consider test contamination, task selection, repeated runs, cost, latency, calibration, and task-specific baselines before making a product claim.

## What the score means

For a text $x$, the model returns a score $s(x)$ associated with a class. When calibrated, a probability estimate describes outcomes under an evaluated distribution; it is not a universal probability that a person or model authored a text. A threshold $t$ converts a score into a decision, such as $\hat y=1$ when $s(x)\ge t$. For a multi-class Choice response, the winning class probability and an API's separate confidence summary are different quantities. A set of independent Noul probabilities is also different from a single normalized distribution. Chapter 07 examines calibration and confidence claims directly.

Calibration can be assessed after training with a held-out calibration set, for example by temperature scaling, or encouraged during training with a proper scoring objective. The article discusses Reinforcement Learning with Calibration Rewards (RLCR) as a published related idea and describes Jev's named Reinforcement Learning for Calibrated Decisions (RLCD) as proprietary. The article does not establish that Jev uses RLCR; keep that distinction when discussing the hypothesis in Chapter 07.

## Data and evaluation contract

The repository describes `rasbt/human-vs-ai-50k` as 25,355 human-written and 25,355 AI-generated texts, with grouped train/validation/test partitions. Row metadata records source and license information. The course treats these as repository-reported dataset facts; inspect the versioned dataset card and row schema before running experiments. Raw text and model weights are distributed separately from Git.

Use training data to fit parameters, validation data to select models and thresholds, and test data once for final reporting. Keep source families together across partitions. For broader claims, add tasks and domains that are meaningfully different from the training distribution. Report split/version, class balance, metric definitions, threshold, sample counts, and uncertainty beside each result. A random row split is not an acceptable substitute when related texts can cross partitions.

A classifier report should include precision, recall, F1, ROC-AUC or PR-AUC as appropriate, a thresholded confusion matrix, calibration evidence, and errors grouped by relevant properties. No one metric establishes safety or generalization. For a general-purpose decision model, one strong result on one dataset does not establish task generality. Test held-out tasks, repeat nondeterministic runs, and check whether benchmark examples may have appeared in training.

## Model lineage and comparison

| Family | Representation and decision rule | What the comparison can teach |
|---|---|---|
| Bag-of-words classifier | Sparse count or TF–IDF features with Naive Bayes, logistic regression, SVM, or boosted trees | A cheap baseline; inspect how loss of word order and dataset shortcuts affect errors. |
| Neural sequence classifier | Word embeddings with recurrent or convolutional layers | How sequence order, local patterns, pooling, and pretraining change the representation. |
| Fine-tuned encoder | Bidirectional contextual representation with a classification head | Strong task-specific classification under a fixed token limit and training setup. |
| Prompted or fine-tuned decoder | Autoregressive model emits a label or uses a task head | Flexible task definition, with generation, masking, cost, and output-format tradeoffs. |
| General-purpose decision API | Instructions and candidate choices/questions/rubrics map to structured decisions | Whether one interface transfers across tasks at useful quality, calibration, speed, and cost. |

Comparisons need clearly separated evidence. Match held-out data, label definitions, tuning budgets, and repeated-run procedures where possible; document mismatches where they cannot be matched. Chapter 08 builds the cost-quality view, while Chapter 10 applies the comparison to a trained, open-weight candidate scorer and task-specific baselines.

## Acceptance conditions and scope

The detector capstone report should let another reader trace labels to sources, reproduce the split, identify the exact model or API version, recreate headline calculations, inspect consequential errors, and understand how to invoke the classifier. The ModernBERT capstone adds a second acceptance condition: the code must turn a state, task instructions, and candidate descriptions into Choice, Noul, and Score responses using a shared candidate-scoring head, and its evaluation must include at least one task held out from training. Report where the model has not been evaluated and distinguish measured results from claims made by a provider or article author.

This course teaches supervised text classification through an AI-text detection case study, then uses a separate multi-task decision model as the final project. It does not establish forensic authorship attribution, certify any detector, or prove that a classifier transfers to arbitrary tasks. Jev's internals and several product claims are not independently available for verification in the article. Scores should not be sole evidence in high-impact decisions about a person or their work.


## Further reading

Sebastian Raschka's [Language Models for Text Classification: From Bag-of-Words to Jev](https://magazine.sebastianraschka.com/p/classifier-history-and-jev) motivates the architecture timeline, Jev-like API forms, calibration discussion, and reported IMDb comparison extended in Chapters 05–08. Its product-specific measurements and hypotheses are presented as dated author-reported evidence, not as independent verification.
